# Downloading Source Data Files

We download the source data files of page view observations from https://dumps.wikimedia.org/other/pageviews/. The data files of January 2024 is specified in the folder **2024-01**, i.e., https://dumps.wikimedia.org/other/pageviews/2024/2024-01/. Since the data is given with each hour, we download 744 data files across 24 hours and different days of January 2024.

In [ ]:
year = 2024
month = 1

import requests
import os
from datetime import datetime, timedelta

def download_specific_pageviews_files(year, month):
    """
    Download pageviews files
    """
    base_url = f"https://dumps.wikimedia.org/other/pageviews/{year:04d}/{year:04d}-{month:02d}/"
    output_dir = "pageviews_data"
    os.makedirs(output_dir, exist_ok=True)
    
    files_to_download = []
    for day in range(1, 32):
        for hour in range(24):
            filename = f"pageviews-{year:04d}{month:02d}{day:02d}-{hour:02d}0000.gz"
            files_to_download.append(filename)
        
    # Download each file
    for filename in files_to_download:
        file_url = base_url + filename
        local_path = os.path.join(output_dir, filename)
        
        try:
            print(f"Downloading: {filename}")
            response = requests.get(file_url)
            response.raise_for_status()
            
            with open(local_path, 'wb') as f:
                f.write(response.content)
            
            print(f"Saved: {local_path}")
            
        except requests.exceptions.RequestException as e:
            print(f"Failed to download {filename}: {e}")

# Run the download
download_specific_pageviews_files(year, month)

# Creating Hourly Page View Time Series



In [ ]:
import pandas as pd
import numpy as np
import time

year = 2024
month = 1

for day in range(1, 32):
    start = time.time()
    print('Processing Day:', day)
    df = pd.read_csv(f'pageviews_data/pageviews-{year:04d}{month:02d}{day:02d}-000000.gz', sep=' ', 
                     names = ['domain_code', 'page_title', f'count_views_{day:02d}00', 'total_response_size'], header = None,
                     on_bad_lines = 'skip', engine = 'python')
    df = df.drop(columns = 'total_response_size')
    df = df.fillna({'domain_code': 'NA', 'page_title': 'NA'})
    df = df.groupby(['domain_code', 'page_title'])[f'count_views_{day:02d}00'].sum().reset_index()
    for hour in range(1, 24):
        df_new = pd.read_csv(f'pageviews_data/pageviews-{year:04d}{month:02d}{day:02d}-{hour:02d}0000.gz', sep=' ', 
                              names = ['domain_code', 'page_title', f'count_views_{day:02d}{hour:02d}', 'total_response_size'], header = None,
                              on_bad_lines = 'skip', engine = 'python')
        df_new = df_new.drop(columns = 'total_response_size')
        df_new = df_new.fillna({'domain_code': 'NA', 'page_title': 'NA'})
        df_new = df_new.groupby(['domain_code', 'page_title'])[f'count_views_{day:02d}{hour:02d}'].sum().reset_index()
        df = pd.merge(df, df_new, on = ['domain_code', 'page_title'], how='outer').fillna(0)
    vec = np.sum(df.iloc[:, 2 :].values, axis = 1)
    df_small = df[vec >= 10].reset_index(drop = True)
    df_small.to_csv(f'data-2024{month:02d}{day:02d}.csv.gz', index = False, compression = 'gzip')
    end = time.time()
    print('Running time (s): ', end - start)
    print()
    print()

In [ ]:
import numpy as np
import pandas as pd

month = 1
day = 1

print('Day:', day)
data = pd.read_csv(f'data-2024{month:02d}{day:02d}.csv.gz', compression='gzip')
df = data[['domain_code', 'page_title']].copy()
df['count_views'] = np.sum(data.iloc[:, 2 :].values, axis = 1)
df = df[df['count_views'] >= 10]
df = df[['domain_code', 'page_title']]
del data

for day in range(2, 32):
    print('Day:', day)
    data_new = pd.read_csv(f'data-2024{month:02d}{day:02d}.csv.gz', compression='gzip')
    df_minimal = data_new[['domain_code', 'page_title']].copy()
    df_minimal['count_views'] = np.sum(data_new.iloc[:, 2 :].values, axis = 1)
    df_minimal = df_minimal[df_minimal['count_views'] >= 10]
    del data_new
    df = pd.merge(df, df_minimal, on = ['domain_code', 'page_title'], how = 'inner')
    df = df[['domain_code', 'page_title']]
df.to_csv(f'data-pages-2024{month:02d}.csv', index = False)

Then, we use the data file of unique pages in e.g., data-pages-202401.csv to index the original data files across 31 days of January 2024. By doing so, we get the data file data-202401.parquet for January 2024.


In [ ]:
import pandas as pd
import numpy as np

month = 1
t = 31

page_ind = pd.read_csv(f'data-pages-2024{month:02d}.csv')
page_ind = page_ind.fillna({'domain_code': 'NA', 'page_title': 'NA'})
df1 = page_ind.iloc[: int(1e+6)]
for day in range(1, t + 1):
    print('Processing Day:', day)
    data = pd.read_csv(f'data-5v-2024{month:02d}{day:02d}.csv.gz', compression='gzip')
    data = data.fillna({'domain_code': 'NA', 'page_title': 'NA'})
    df1 = pd.merge(df1, data, on=['domain_code', 'page_title'], how='left')
    del data

df2 = page_ind.iloc[int(1e+6) : int(2e+6)]
for day in range(1, t + 1):
    print('Processing Day:', day)
    data = pd.read_csv(f'data-5v-2024{month:02d}{day:02d}.csv.gz', compression='gzip')
    data = data.fillna({'domain_code': 'NA', 'page_title': 'NA'})
    df2 = pd.merge(df2, data, on=['domain_code', 'page_title'], how='left')
    del data

df3 = page_ind.iloc[int(2e+6) :]
for day in range(1, t + 1):
    print('Processing Day:', day)
    data = pd.read_csv(f'data-5v-2024{month:02d}{day:02d}.csv.gz', compression='gzip')
    data = data.fillna({'domain_code': 'NA', 'page_title': 'NA'})
    df3 = pd.merge(df3, data, on=['domain_code', 'page_title'], how='left')
    del data

df = pd.concat([df1, df2, df3], ignore_index=True)
df.to_parquet(f'data-2024{month:02d}.parquet', compression='zstd')